# CDS 303 Final Project: Data Reference Notebook

**Team 8:** Dae Yeong Mun, Oscar Molina Romualdo



Datasets and their sources are listed on the project website: https://oscarmvr-12.github.io/Seoul-Bike/supplementary-data.html


In [1]:
import pandas as pd

### English Translation Setup

The source files from the Seoul Open Data Portal use Korean column names, and a few columns use Korean category values or Korean place names. Everything below is translated to English immediately after loading, so every table and output in this notebook — from here on — is in English.

**Columns**

| Korean | English |
|---|---|
| 자전거번호 | bike_id |
| 대여일시 | rental_datetime |
| 대여 대여소번호 | rental_station_number |
| 대여 대여소명 | rental_station_name_kr |
| 대여거치대 | rental_dock_number |
| 반납일시 | return_datetime |
| 반납대여소번호 | return_station_number |
| 반납대여소명 | return_station_name_kr |
| 반납거치대 | return_dock_number |
| 이용시간(분) | duration_min |
| 이용거리(M) | distance_m |
| 생년 | birth_year |
| 성별 | gender |
| 이용자종류 | user_type |
| 대여대여소ID | rental_station_id |
| 반납대여소ID | return_station_id |
| 대여일자 (daily summary file) | rental_date |
| 대여건수 (daily summary file) | rental_count |

**Category values**: 내국인 → Domestic, 외국인 → Foreigner, 비회원 → Non-member; M/m → Male, F/f → Female.

**Station names** are Korean place names (subway exits, landmarks), so rather than a meaning-based "translation" we romanize them — the same Korean-to-Latin-alphabet spelling system used on Korean street signs — computed directly from the Hangul characters below (no external service, works for all 2,761 stations).

In [2]:
## English column-name and category-value mappings
COLUMN_TRANSLATIONS = {
    "자전거번호": "bike_id",
    "대여일시": "rental_datetime",
    "대여 대여소번호": "rental_station_number",
    "대여 대여소명": "rental_station_name_kr",
    "대여거치대": "rental_dock_number",
    "반납일시": "return_datetime",
    "반납대여소번호": "return_station_number",
    "반납대여소명": "return_station_name_kr",
    "반납거치대": "return_dock_number",
    "이용시간(분)": "duration_min",
    "이용거리(M)": "distance_m",
    "생년": "birth_year",
    "성별": "gender",
    "이용자종류": "user_type",
    "대여대여소ID": "rental_station_id",
    "반납대여소ID": "return_station_id",
    "대여일자": "rental_date",
    "대여건수": "rental_count",
    "지점": "station_id",
    "지점명": "station_name_kr",
    "일시": "date",
    "평균기온(°C)": "avg_temp_c",
    "최저기온(°C)": "min_temp_c",
    "최고기온(°C)": "max_temp_c",
    "일강수량(mm)": "precip_mm",
    "평균 풍속(m/s)": "avg_wind_mps",
    "평균 상대습도(%)": "avg_humidity_pct",
    "합계 일조시간(hr)": "sunshine_hr",
}

GENDER_MAP = {"M": "Male", "m": "Male", "F": "Female", "f": "Female"}
USER_TYPE_MAP = {"내국인": "Domestic", "외국인": "Foreigner", "비회원": "Non-member"}

def to_english(df):
    """Rename columns to English and translate gender/user_type values, if present."""
    df = df.rename(columns=COLUMN_TRANSLATIONS)
    if "gender" in df.columns:
        df["gender"] = df["gender"].map(GENDER_MAP).fillna(df["gender"])
    if "user_type" in df.columns:
        df["user_type"] = df["user_type"].map(USER_TYPE_MAP).fillna(df["user_type"])
    return df

In [3]:
## Romanize Korean station names (Revised Romanization, pure Python)
CHO = ["g","kk","n","d","tt","r","m","b","pp","s","ss","","j","jj","ch","k","t","p","h"]
JUNG = ["a","ae","ya","yae","eo","e","yeo","ye","o","wa","wae","oe","yo","u","wo","we","wi","yu","eu","ui","i"]
JONG = ["","k","k","k","n","n","n","t","l","k","m","l","l","l","p","l","m","p","p","t","t","ng","t","t","k","t","p","t"]

def romanize_syllable(ch):
    code = ord(ch)
    if 0xAC00 <= code <= 0xD7A3:
        idx = code - 0xAC00
        return CHO[idx // (21*28)] + JUNG[(idx % (21*28)) // 28] + JONG[idx % 28]
    return ch

def romanize(text):
    if pd.isna(text):
        return text
    return "".join(romanize_syllable(c) for c in str(text)).title()

## Daily Rental Summary

In [4]:
df = to_english(pd.read_csv("서울특별시 공공자전거 일별 대여건수_26.1-6.csv", encoding="cp949"))

In [5]:
## First Few Rows
df.head()

,rental_date,rental_count
0,2026-01-01,27245
1,2026-01-02,47606
2,2026-01-03,41385
3,2026-01-04,42743
4,2026-01-05,68116


In [6]:
## Dataset Dimensions
df.shape

(181, 2)

In [7]:
## Column Names
df.columns

Index(['rental_date', 'rental_count'], dtype='str')

In [8]:
## Data Types
df.dtypes

rental_date       str
rental_count    int64
dtype: object

In [9]:
## Basic Summary Statistics
df.describe()

,rental_count
count,181.000000
mean,99331.441989
std,41517.757494
min,9975.000000
25%,62125.000000
50%,102124.000000
75%,138004.000000
max,158650.000000


In [10]:
## Missing Values
df.isnull().sum()

rental_date     0
rental_count    0
dtype: int64

## Trip-Level Rental History (Jan–Jun 2026)

The main dataset: one combined CSV of about 18.5 million individual Ttareungi trips (~3.1 GB, CP949-encoded) from Seoul Open Data Plaza (OA-15182). Each row records the rental and return station, timestamps, trip duration and distance, and rider details.

The file is too large to load comfortably in one go, so later analysis should read it in chunks (`pd.read_csv(..., chunksize=...)`) with only the columns needed (`usecols=`).

**Filename note:** macOS stores Korean filenames in Unicode NFD form, while text typed in code is usually NFC. The two look identical but don't match, so a plain `glob` finds nothing. The cell below normalizes both sides before comparing.


In [11]:
import os
import unicodedata

def find_files(prefix, suffix=".csv", folder="."):
    """Find files whose names start with `prefix`, ignoring NFC/NFD Unicode differences."""
    norm = lambda s: unicodedata.normalize("NFC", s)
    return sorted(
        os.path.join(folder, f) for f in os.listdir(folder)
        if norm(f).startswith(norm(prefix)) and f.endswith(suffix)
    )

trip_files = find_files("서울특별시 공공자전거 대여이력 정보_")
print(f"Found {len(trip_files)} trip file(s):", [unicodedata.normalize("NFC", os.path.basename(f)) for f in trip_files])


Found 1 trip file(s): ['서울특별시 공공자전거 대여이력 정보_전체_2601-2606.csv']


### English Station Lookup

Maps each station ID to its station number, Korean name and romanized English name. If `station_lookup_en.csv` is missing, it is rebuilt from the trip file.

**Note:** the existing lookup has 2,761 stations because it was built from a single month of trips. `station_locations.csv` has 3,430. Future joins between trips and coordinates should use **station IDs** (`rental_station_id` ↔ `대여소_ID`), not station names.


In [12]:
## English station lookup: station ID, station number, Korean name, romanized English name
import os

if os.path.exists("station_lookup_en.csv"):
    stations_en = pd.read_csv("station_lookup_en.csv")
else:
    _stations_raw = pd.read_csv(
        trip_files[-1], encoding="cp949",
        usecols=["대여대여소ID", "대여 대여소번호", "대여 대여소명"],
    ).drop_duplicates(subset=["대여대여소ID"])
    stations_en = _stations_raw.rename(columns={
        "대여대여소ID": "station_id",
        "대여 대여소번호": "station_number",
        "대여 대여소명": "station_name_kr",
    })
    stations_en["station_name_en"] = stations_en["station_name_kr"].apply(romanize)
    stations_en = stations_en.sort_values("station_number")
    stations_en.to_csv("station_lookup_en.csv", index=False)

kr_to_en_station = dict(zip(stations_en["station_name_kr"], stations_en["station_name_en"]))
print(f"{len(stations_en)} stations")
stations_en.head(10)

2761 stations


,station_number,station_name_kr,station_id,station_name_en
0,102,망원역 1번출구 앞,ST-4,Mangwonyeok 1Beonchulgu Ap
1,103,망원역 2번출구 앞,ST-5,Mangwonyeok 2Beonchulgu Ap
2,104,합정역 1번출구 앞,ST-6,Hapjeongyeok 1Beonchulgu Ap
3,105,합정역 5번출구 앞,ST-7,Hapjeongyeok 5Beonchulgu Ap
4,106,합정역 7번출구 앞,ST-8,Hapjeongyeok 7Beonchulgu Ap
5,107,수협 연남동지점,ST-9,Suhyeop Yeonnamdongjijeom
6,108,서교동 사거리,ST-10,Seogyodong Sageori
7,111,상수역 2번출구 앞,ST-15,Sangsuyeok 2Beonchulgu Ap
8,113,홍대입구역 2번출구 앞,ST-18,Hongdaeipguyeok 2Beonchulgu Ap
9,114,홍대입구역 8번출구 앞,ST-20,Hongdaeipguyeok 8Beonchulgu Ap


## Secondary Dataset: Seoul Daily Weather (Jan–Jun 2026)

To study how weather conditions affect bike rental demand, we use daily surface weather observations for Seoul from the Korea Meteorological Administration (KMA) Open Data Portal ([data.kma.go.kr](https://data.kma.go.kr)), Automated Synoptic Observing System (ASOS), station 108 (Seoul).

The data covers the same date range as the bike rental data, **January 1 - June 30, 2026** (181 days), and includes: average/minimum/maximum temperature, daily precipitation, average wind speed, average relative humidity, and total sunshine hours.

Note: `precip_mm` is blank in the raw KMA data on days with no measurable precipitation; we fill these with 0 below.


**Note:** this is a single station (108) observing the whole city once a day, so it gives city-wide conditions, not local ones.


In [13]:
## Load and translate the daily weather dataset
weather = to_english(pd.read_csv("seoul_weather_2026_01-06.csv", encoding="cp949"))
weather["date"] = pd.to_datetime(weather["date"])
weather["precip_mm"] = weather["precip_mm"].fillna(0)
weather.head()


,station_id,station_name_kr,date,avg_temp_c,min_temp_c,max_temp_c,precip_mm,avg_wind_mps,avg_humidity_pct,sunshine_hr
0,108,서울,2026-01-01,-7.4,-10.5,-2.1,0.0,2.8,41.5,8.9
1,108,서울,2026-01-02,-8.0,-11.4,-3.8,0.0,2.7,46.6,9.1
2,108,서울,2026-01-03,-4.7,-9.8,0.0,0.0,1.4,60.0,4.5
3,108,서울,2026-01-04,0.2,-2.7,5.0,0.0,1.9,63.4,7.4
4,108,서울,2026-01-05,-1.9,-4.6,1.8,0.0,2.3,48.5,9.1


In [14]:
## Weather summary statistics
weather.describe()


,station_id,date,avg_temp_c,min_temp_c,max_temp_c,precip_mm,avg_wind_mps,avg_humidity_pct,sunshine_hr
count,181.0,181,181.000000,181.000000,181.000000,181.000000,181.000000,181.000000,181.000000
mean,108.0,2026-04-01 00:00:00,11.030939,6.255249,16.525414,1.623757,2.336464,55.579006,7.573481
min,108.0,2026-01-01 00:00:00,-9.700000,-13.200000,-5.200000,0.000000,1.100000,30.300000,0.000000
25%,108.0,2026-02-15 00:00:00,2.700000,-1.300000,7.500000,0.000000,1.900000,46.600000,4.500000
50%,108.0,2026-04-01 00:00:00,11.900000,7.100000,18.000000,0.000000,2.200000,53.900000,8.900000
75%,108.0,2026-05-16 00:00:00,20.600000,15.100000,26.400000,0.000000,2.600000,63.400000,10.700000
max,108.0,2026-06-30 00:00:00,27.200000,23.000000,34.000000,71.700000,5.300000,90.500000,13.500000
std,0.0,NaN,10.301792,10.017210,10.890986,7.600237,0.584044,12.790309,3.989398


## Spatial Datasets

The four datasets planned for the urban-design phase have now been collected and are loaded below:

1. **Seoul District Boundaries** (`seoul_district_boundaries.geojson`) - 25 district polygons with Korean/English names, sourced from a maintained GitHub mirror (`datainworld/administrative_district`) since Seoul Open Data Plaza had no direct download for this.
2. **Subway Station Coordinates** (`subway_station_locations.csv`) - 784 stations with GPS coordinates, from data.seoul.go.kr (OA-21232).
3. **Seoul Bike Path Network** (`seoul_bike_path_network.geojson`) - 3,535 cycleway segments, pulled from OpenStreetMap via Overpass Turbo since Seoul only publishes fragmented district-level bike-path files, not a citywide one.
4. **Floating Population Estimates** (`seoul_floating_population_2026_01-06.csv`) - daily population estimates by district, Jan-Jun 2026 (4,706 rows), combining domestic and foreign (long/short-term) residents in one dataset.

Also collected: real Ttareungi station GPS coordinates (`station_locations.csv`, 3,430 stations).


### Loading the Spatial Datasets


In [15]:
## Ttareungi station GPS coordinates
stations_geo = pd.read_csv("station_locations.csv", encoding="cp949")
stations_geo = stations_geo.rename(columns={
    "대여소_ID": "station_id",
    "주소1": "address_1",
    "주소2": "address_2",
    "위도": "lat",
    "경도": "lon",
})
print("shape:", stations_geo.shape)
print(stations_geo.dtypes)
print("rows with missing/zero coordinates:", ((stations_geo["lat"] == 0) & (stations_geo["lon"] == 0)).sum())
stations_geo.head()


shape: (3430, 5)
station_id        str
address_1         str
address_2         str
lat           float64
lon           float64
dtype: object
rows with missing/zero coordinates: 77


,station_id,address_1,address_2,lat,lon
0,ST-999,서울특별시 양천구 목동서로 280,목동아파트 8단지 상가동,0.000000,0.000000
1,ST-998,서울특별시 양천구 목동서로 130,목동아파트 4단지 상가동,0.000000,0.000000
2,ST-997,서울특별시 양천구 목동중앙로 49,목동3단지 시내버스정류장,37.534390,126.869598
3,ST-996,서울특별시 양천구 남부순환로88길5-16,양강중학교앞 교차로,37.524334,126.850548
4,ST-995,서울특별시 양천구 중앙로 153 공중화장실,NaN,37.510597,126.857323


**Note:** 77 stations have (0, 0) coordinates and must be excluded from any spatial feature.


In [16]:
## Subway station coordinates
subway = pd.read_csv("subway_station_locations.csv", encoding="cp949")
subway = subway.rename(columns={
    "역사_ID": "station_id",
    "역사명": "station_name",
    "호선": "line",
    "위도": "lat",
    "경도": "lon",
})
print("shape:", subway.shape)
print(subway.dtypes)
print("missing values:\n", subway.isnull().sum())
subway.head()


shape: (784, 5)
station_id        int64
station_name        str
line                str
lat             float64
lon             float64
dtype: object
missing values:
 station_id      0
station_name    0
line            0
lat             0
lon             0
dtype: int64


,station_id,station_name,line,lat,lon
0,9010,동탄,수도권 광역급행철도,37.20034,127.09569
1,9009,구성,수도권 광역급행철도,37.29913,127.10389
2,9008,성남,수도권 광역급행철도,37.39467,127.12058
3,9007,수서,수도권 광역급행철도,37.48637,127.10161
4,9006,삼성,수도권 광역급행철도,37.50887,127.06324


In [17]:
## Seoul district boundaries (GeoJSON)
import json

with open("seoul_district_boundaries.geojson", encoding="utf-8") as f:
    district_geo = json.load(f)

print("feature count:", len(district_geo["features"]))
district_names = [feat["properties"]["SIG_ENG_NM"] for feat in district_geo["features"]]
print("districts:", sorted(district_names))


feature count: 25
districts: ['Dobong-gu', 'Dongdaemun-gu', 'Dongjak-gu', 'Eunpyeong-gu', 'Gangbuk-gu', 'Gangdong-gu', 'Gangnam-gu', 'Gangseo-gu', 'Geumcheon-gu', 'Guro-gu', 'Gwanak-gu', 'Gwangjin-gu', 'Jongno-gu', 'Jung-gu', 'Jungnang-gu', 'Mapo-gu', 'Nowon-gu', 'Seocho-gu', 'Seodaemun-gu', 'Seongbuk-gu', 'Seongdong-gu', 'Songpa-gu', 'Yangcheon-gu', 'Yeongdeungpo-gu', 'Yongsan-gu']


In [18]:
## Seoul bike path network (GeoJSON, from OpenStreetMap)
with open("seoul_bike_path_network.geojson", encoding="utf-8") as f:
    bike_path_geo = json.load(f)

print("feature count:", len(bike_path_geo["features"]))
geom_types = pd.Series([feat["geometry"]["type"] for feat in bike_path_geo["features"] if feat.get("geometry")]).value_counts()
print("geometry types:\n", geom_types)

## Sample of tagged attributes (name, operator, distance in km)
bike_path_props = pd.DataFrame([feat["properties"] for feat in bike_path_geo["features"]])
bike_path_props[["name", "operator", "distance"]].head(10)


feature count: 3536
geometry types:
 LineString    3535
Point            1
Name: count, dtype: int64


,name,operator,distance
0,위례성대로(남측),서울특별시 동부도로사업소,0.208
1,위례성대로(남측),송파구 도시교통과,0.163
2,양재대로,송파구 도시교통과,0.539
3,중랑천서자전거길,NaN,NaN
4,중랑천서자전거길,NaN,NaN
5,중랑천동자전거길,NaN,NaN
6,압구정 나들목,NaN,NaN
7,염창나들목,NaN,NaN
8,중랑천 자전거길 (우이천 합수부),NaN,NaN
9,안양천 자전거길,NaN,NaN


In [19]:
## Floating population estimates (daily, by district, Jan-Jun 2026)
floating_pop = to_english(pd.read_csv("seoul_floating_population_2026_01-06.csv", encoding="utf-8-sig"))
print("shape:", floating_pop.shape)
print(floating_pop.dtypes)
print("missing values:\n", floating_pop.isnull().sum())
floating_pop.head()


shape: (4706, 14)
기준일ID             int64
시구코드              int64
시구명                 str
총생활인구수          float64
내국인생활인구수        float64
장기체류외국인인구수      float64
단기체류외국인인구수      float64
일최대인구수          float64
일최소인구수          float64
주간인구수(09~18)    float64
야간인구수(19~08)    float64
서울외유입인구수        float64
자치구간이동인구수       float64
적재일시              int64
dtype: object
missing values:
 기준일ID           0
시구코드            0
시구명             0
총생활인구수          0
내국인생활인구수        0
장기체류외국인인구수      0
단기체류외국인인구수      0
일최대인구수          0
일최소인구수          0
주간인구수(09~18)    0
야간인구수(19~08)    0
서울외유입인구수        0
자치구간이동인구수       0
적재일시            0
dtype: int64


,기준일ID,시구코드,시구명,총생활인구수,내국인생활인구수,장기체류외국인인구수,단기체류외국인인구수,일최대인구수,일최소인구수,주간인구수(09~18),야간인구수(19~08),서울외유입인구수,자치구간이동인구수,적재일시
0,20260131,11000,서울시,1.143818e+07,1.096621e+07,315876.95616,156099.69409,12450290.55,10654861.65,1.187774e+07,1.112421e+07,1157061.91,2046290.14,20260727
1,20260131,11110,종로구,2.728191e+05,2.524318e+05,12033.35666,8353.85791,410000.27,189673.31,3.436303e+05,2.222396e+05,61411.11,150744.43,20260727
2,20260131,11140,중구,2.592668e+05,2.205160e+05,15726.47166,23024.32625,368309.45,183448.18,3.221502e+05,2.143500e+05,37070.18,139501.36,20260727
3,20260131,11170,용산구,3.148673e+05,2.825674e+05,25773.59041,6526.27583,375403.66,265446.29,3.432500e+05,2.945939e+05,57303.87,87739.95,20260727
4,20260131,11200,성동구,3.430844e+05,3.270154e+05,10896.26500,5172.71750,368714.94,324586.26,3.584763e+05,3.320902e+05,29397.78,79887.00,20260727


**Note:** each day has 26 rows, the 25 districts plus a Seoul-wide total (`시구코드` = 11000). Remove the total row before summing or joining by district. The column names are still in Korean because they aren't in `COLUMN_TRANSLATIONS` yet.
